# fase_1 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 1.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

## 3. Helper

In [3]:
from IPython.display import display

def fetch_df(query):
    return pd.read_sql(query, db_old)

def check_nulls(df):
    print("\nNULL CHECK:")
    display(df.isnull().sum())

def check_duplicates(df, subset_cols):
    dup = df[df.duplicated(subset=subset_cols)]
    print(f"\nDUPLICATE ROWS: {len(dup)}")
    display(dup.head())

def preview_df(df, title="Preview", limit=10):
    print(f"\n{title}:")
    display(df.head(limit))

def compare_count(table_old, table_new):
    old = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_old}", db_old)['total'][0]
    new = pd.read_sql(f"SELECT COUNT(*) as total FROM {table_new}", db_new)['total'][0]

    print(f"\nCOUNT CHECK → OLD: {old} | NEW: {new}")
    print("STATUS:", "OK ✅" if old == new else "CHECK ⚠️")

def check_dtype(df, table_name):
    print(f"\n=== CHECK TIPE DATA: {table_name} ===")

    db_schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for col in df.columns:
        df_type = df[col].dtype

        db_type = db_schema[db_schema['Field'] == col]['Type'].values
        db_type = db_type[0] if len(db_type) > 0 else "NOT FOUND"

        print(f"{col} → DF: {df_type} | DB: {db_type}")

In [4]:
def report_not_null_violations(df, table_name):
    print(f"\n=== NOT NULL VIOLATION: {table_name} ===")

    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    violations = {}

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']

        if col in df.columns and is_nullable == 'NO':
            null_count = df[col].isnull().sum()

            if null_count > 0:
                violations[col] = null_count

    if not violations:
        print("✅ Semua kolom NOT NULL aman")
        return False
    else:
        print("⚠️ Kolom NOT NULL yang bermasalah:")
        for k, v in violations.items():
            print(f"{k}: {v} NULL")
        return True
    
def enforce_not_null(df, table_name):
    schema = pd.read_sql(f"DESCRIBE {table_name}", db_new)

    for _, row in schema.iterrows():
        col = row['Field']
        is_nullable = row['Null']
        col_type = row['Type']

        if col in df.columns and is_nullable == 'NO':
            if df[col].isnull().sum() > 0:

                if 'int' in col_type:
                    df[col] = df[col].fillna(0)
                elif 'date' in col_type or 'time' in col_type:
                    df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
                else:
                    df[col] = df[col].fillna('Unknown')

    return df


In [5]:
def print_summary(df, table_name):
    """Menampilkan ringkasan dataframe setelah transform"""
    
    print(f"\n{'='*60}")
    print(f"📊 RINGKASAN: {table_name.upper()}")
    print(f"{'='*60}")
    
    print(f"\n📈 DIMENSI DATA:")
    print(f"   • Baris: {len(df):,}")
    print(f"   • Kolom: {len(df.columns)}")
    
    print(f"\n📋 KOLOM: {list(df.columns)}")
    
    print(f"\n🔍 TIPE DATA:")
    for col in df.columns:
        print(f"   • {col}: {df[col].dtype}")
    
    print(f"\n⚠️  NULL COUNT:")
    null_info = df.isnull().sum()
    has_null = False
    for col, count in null_info.items():
        if count > 0:
            print(f"   • {col}: {count} NULL")
            has_null = True
    if not has_null:
        print(f"   ✅ Tidak ada NULL")
    
    print(f"\n📐 STATISTIK:")
    print(f"   • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    
    print(f"\n👀 PREVIEW DATA (5 baris pertama):")
    display(df.head())
    print(f"{'='*60}\n")

## 4. Migrating

### Cara Menampilkan Tabel DataFrame

Ada beberapa cara untuk menampilkan dataframe sebagai tabel di Jupyter Notebook:

1. **`display(df)`** - Menampilkan tabel HTML yang rapi ✅ (Recommended)
2. **`df`** - Menampilkan last expression (hanya jika di akhir cell)
3. **`df.head()`** - Menampilkan 5 baris pertama
4. **`df.tail()`** - Menampilkan 5 baris terakhir
5. **`print(df.to_string())`** - Menampilkan sebagai text (kurang bagus)
6. **`df.to_html()`** - Menghasilkan HTML string


In [6]:
from IPython.display import display

def migrate_kursus():
    print("\n=== MIGRATING KURSUS ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idpendkursus, nama_kursus, keterangan
        FROM pendidikankursus
    """)

    print("\n📥 DATA ASLI")
    display(df_old.head())

    # 2. TRANSFORM
    df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'nama_kursus': 'nama_kursus',
        'keterangan': 'deskripsi'
    })

    # 3. CLEANING
    df['deskripsi'] = df['deskripsi'].fillna('')
    df['id_kursus'] = df['id_kursus'].astype(str)

    # 4. VALIDASI
    check_dtype(df, "kursus")
    check_nulls(df)
    check_duplicates(df, ['id_kursus'])

    # 5. PREVIEW HASIL TRANSFORM
    print("\n📤 SETELAH TRANSFORM")
    display(df.head())

    return df

In [7]:
def migrate_level():
    print("\n=== MIGRATING level ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idlevel, level, tingkatan
        FROM level
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM (mapping kolom)
    df = df_old.rename(columns={
        'idlevel': 'id_level',
        'level': 'nama_level',
        'tingkatan': 'urutan_level'
    })

    # 3. CLEANING

    ## handle null
    df['nama_level'] = df['nama_level'].fillna('Unknown')

    # cek pelanggaran NOT NULL
    violation = report_not_null_violations(df, "level")

    ## pastikan tipe
    df['id_level'] = df['id_level'].astype(str)
    check_dtype(df, "level")

    # 4. VALIDASI
    check_nulls(df)
    check_duplicates(df, ['id_level'])

    # cek data desimal di urutan_level
    display(df[df['urutan_level'] % 1 != 0])

    # fix tipe urutan_level
    df['urutan_level'] = df['urutan_level'].fillna(0)
    df['urutan_level'] = df['urutan_level'].astype(int)

    check_dtype(df, "level")

    # 5. PREVIEW HASIL TRANSFORM
    preview_df(df, "SETELAH TRANSFORM")

    return df

In [8]:
from IPython.display import display

def migrate_sesi():
    print("\n=== MIGRATING SESI ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idsesi, nama_sesi, waktu_awal, waktu_akhir
        FROM sesi
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM
    df = df_old.rename(columns={
        'idsesi': 'id_sesi',
        'nama_sesi': 'nama_sesi',
        'waktu_awal': 'waktu_mulai',
        'waktu_akhir': 'waktu_selesai'
    })

    # 3. CLEANING

    ## handle null
    df['nama_sesi'] = df['nama_sesi'].fillna('Tidak Ada')

    ## strip spasi
    df['nama_sesi'] = df['nama_sesi'].str.strip()

    ## tipe id
    df['id_sesi'] = df['id_sesi'].astype(str)

    ## handle waktu (biar sesuai TIME / DATETIME)
    df['waktu_mulai'] = pd.to_timedelta(df['waktu_mulai']).dt.components.apply(
    lambda x: f"{int(x.hours):02d}:{int(x.minutes):02d}:{int(x.seconds):02d}", axis=1
    )

    df['waktu_selesai'] = pd.to_timedelta(df['waktu_selesai']).dt.components.apply(
    lambda x: f"{int(x.hours):02d}:{int(x.minutes):02d}:{int(x.seconds):02d}", axis=1
    )

    # 4. VALIDASI NOT NULL
    violation = report_not_null_violations(df, "sesi")

    if violation:
        print("\n⚠️ APPLY ENFORCE NOT NULL")
        df = enforce_not_null(df, "sesi")

    # 5. VALIDASI UMUM
    check_dtype(df, "sesi")
    check_nulls(df)
    check_duplicates(df, ['id_sesi'])

    # 6. PREVIEW
    preview_df(df, "SETELAH TRANSFORM")

    return df

In [9]:
def migrate_libur():
    print("\n=== MIGRATING LIBUR ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idlibur, title, start, end
        FROM libur
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM (mapping kolom)
    df = df_old.rename(columns={
        'idlibur': 'id_libur',
        'title': 'nama_event',
        'start': 'tanggal_mulai',
        'end': 'tanggal_berakhir'
    })

    # 3. CLEANING

    ## handle null
    df['nama_event'] = df['nama_event'].fillna('Tidak Ada')

    ## TAMBAHKAN SEMUA KOLOM BARU DARI DB BARU
    df['deskripsi_libur'] = ''
    df['sumber'] = ''
    df['label_warna'] = ''
    df['status_libur_program'] = 1

    # cek pelanggaran NOT NULL
    violation = report_not_null_violations(df, "libur")

    if violation:
        print("\n⚠️ APPLY ENFORCE NOT NULL")
        df = enforce_not_null(df, "libur")

    # 4. VALIDASI

    ## tipe data
    df['id_libur'] = df['id_libur'].astype(str)
    check_dtype(df, "libur")

    ## null & duplicate
    check_nulls(df)
    check_duplicates(df, ['id_libur'])

    # 5. PREVIEW HASIL TRANSFORM
    preview_df(df, "SETELAH TRANSFORM")

    return df

In [10]:
def migrate_topik_diskusi():
    print("\n=== MIGRATING TOPIK DISKUSI ===")

    # 1. EXTRACT
    df_old = fetch_df("""
        SELECT idtagmd, tag
        FROM tag_materi_diskusi
    """)

    preview_df(df_old, "DATA ASLI")

    # 2. TRANSFORM
    df = df_old.rename(columns={
        'idtagmd': 'id_topik_diskusi',
        'tag': 'topik_diskusi'
    })

    # ⚠️ ID akan di-auto increment → jangan dipakai saat insert
    df['id_topik_diskusi'] = None

        # 3. CLEANING

    ## handle null
    df['topik_diskusi'] = df['topik_diskusi'].fillna('Tidak Ada')

    ## bersihin spasi
    df['topik_diskusi'] = df['topik_diskusi'].str.strip()

    ## tambahkan kolom yang ada di DB baru
    df['deskripsi_topik_diskusi'] = ''

    # 4. REMOVE DUPLICATE (berdasarkan isi topik)
    df = df.drop_duplicates(subset=['topik_diskusi'])

    # ❗ JANGAN VALIDASI NOT NULL UNTUK ID, tapi jangan drop kolom deskripsi
    non_id_cols = [col for col in df.columns if col != 'id_topik_diskusi']
    violation = report_not_null_violations(df[non_id_cols], "topik_diskusi")
    
    if violation:
        print("\n⚠️ APPLY ENFORCE NOT NULL (NON-ID)")
        df_non_id = enforce_not_null(df.drop(columns=['id_topik_diskusi']), "topik_diskusi")
        df['topik_diskusi'] = df_non_id['topik_diskusi']

    # ❗ JANGAN ASTYPE ID
    check_dtype(df.drop(columns=['id_topik_diskusi']), "topik_diskusi")

    ## null & duplicate
    check_nulls(df)
    check_duplicates(df, ['topik_diskusi'])

    # 7. PREVIEW HASIL FINAL
    preview_df(df, "SETELAH TRANSFORM")

    return df

In [11]:
def prepare_kursus_level():
    print("\n=== PREPARE KURSUS LEVEL ===")

    df_old = fetch_df("""
        SELECT idpendkursus, idlevel
        FROM level
    """)

    df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'idlevel': 'id_level'
    })

    # cleaning
    df['id_kursus'] = df['id_kursus'].astype(str)
    df['id_level'] = df['id_level'].astype(str)  # karena kamu simpan string

    df = df.dropna(subset=['id_kursus', 'id_level'])
    df = df.drop_duplicates(subset=['id_kursus', 'id_level'])

    preview_df(df, "KURSUS LEVEL SIAP INSERT")

    return df

In [12]:
def prepare_kursus_libur():
    print("\n=== PREPARE KURSUS LIBUR ===")

    df_old = fetch_df("""
        SELECT idpendkursus, idlibur
        FROM libur_pendkursus
    """)

    df = df_old.rename(columns={
        'idpendkursus': 'id_kursus',
        'idlibur': 'id_libur'
    })

    df['id_kursus'] = df['id_kursus'].astype(str)
    df['id_libur'] = df['id_libur'].astype(str)

    df = df.dropna(subset=['id_kursus', 'id_libur'])
    df = df.drop_duplicates(subset=['id_kursus', 'id_libur'])

    preview_df(df, "KURSUS LIBUR SIAP INSERT")
    return df

In [13]:
# RINGKASAN SEMUA TRANSFORMASI
print("\n" + "="*60)
print("🎯 SUMMARY TRANSFORMASI DATA FASE 1")
print("="*60)

# 1. Kursus
print("\n[1/5] Migrasi KURSUS")
df_kursus = migrate_kursus()
print_summary(df_kursus, "kursus")

# 2. Level
print("\n[2/5] Migrasi LEVEL")
df_level = migrate_level()
print_summary(df_level, "level")

# 3. Sesi
print("\n[3/5] Migrasi SESI")
df_sesi= migrate_sesi()
print_summary(df_sesi, "sesi")

# 4. Libur
print("\n[4/5] Migrasi LIBUR")
df_libur = migrate_libur()
print_summary(df_libur, "libur")

# 5. Topik Diskusi
print("\n[5/5] Migrasi TOPIK DISKUSI")
df_topik = migrate_topik_diskusi()
print_summary(df_topik, "topik_diskusi")

# . Kursus Level
print("\n[6/5] Migrasi KURSUS LEVEL")
df_kursus_level = prepare_kursus_level()
print_summary(df_kursus_level, "kursus_level")

# . Kursus Libur
print("\n[6/5] Migrasi KURSUS LIBUR")
df_kursus_libur = prepare_kursus_libur()
print_summary(df_kursus_libur, "kursus_libur")

print("\n✅ SEMUA TRANSFORMASI SELESAI")


🎯 SUMMARY TRANSFORMASI DATA FASE 1

[1/5] Migrasi KURSUS

=== MIGRATING KURSUS ===

📥 DATA ASLI


,idpendkursus,nama_kursus,keterangan
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner"
1,K00002,LEAP - Coding Class,Coding Class Regular
2,K00003,LEAP - Leap Literacy Club,LLC
3,K00004,LEAP - Conversation Class,Conversation Class for Adults
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,"



=== CHECK TIPE DATA: kursus ===
id_kursus → DF: object | DB: varchar(15)
nama_kursus → DF: object | DB: varchar(150)
deskripsi → DF: object | DB: text

NULL CHECK:


id_kursus      0
nama_kursus    0
deskripsi      0
dtype: int64


DUPLICATE ROWS: 0


...



[6/5] Migrasi KURSUS LIBUR

=== PREPARE KURSUS LIBUR ===

KURSUS LIBUR SIAP INSERT:


,id_kursus,id_libur
0,K00001,L00070
1,K00001,L00068



📊 RINGKASAN: KURSUS_LIBUR

📈 DIMENSI DATA:
   • Baris: 2
   • Kolom: 2

📋 KOLOM: ['id_kursus', 'id_libur']

🔍 TIPE DATA:
   • id_kursus: object
   • id_libur: object

⚠️  NULL COUNT:
   ✅ Tidak ada NULL

📐 STATISTIK:
   • Memory usage: 0.34 KB

👀 PREVIEW DATA (5 baris pertama):


,id_kursus,id_libur
0,K00001,L00070
1,K00001,L00068




✅ SEMUA TRANSFORMASI SELESAI


In [14]:
df_fase1_afrida = {
    "kursus": df_kursus,
    "level": df_level,
    "sesi": df_sesi,
    "libur": df_libur,
    "topik_diskusi": df_topik,
    "kursus_level": df_kursus_level,
    "kursus_libur": df_kursus_libur
}

In [15]:
import pickle

file_path = "fase_1_afrida.pkl"

with open(file_path, "wb") as f:
    pickle.dump(df_fase1_afrida, f)

print(f"✅ Data berhasil disimpan ke {file_path}")

✅ Data berhasil disimpan ke fase_1_afrida.pkl


In [16]:
with open("fase_1_afrida.pkl", "rb") as f:
    data_loaded = pickle.load(f)

print("📦 Isi file:")
for key in data_loaded.keys():
    print(f" - {key}: {data_loaded[key].shape}")

📦 Isi file:
 - kursus: (21, 3)
 - level: (181, 3)
 - sesi: (43, 4)
 - libur: (79, 8)
 - topik_diskusi: (11, 3)
 - kursus_level: (181, 2)
 - kursus_libur: (2, 2)


In [17]:
print("Shape:", df_sesi.shape)
print("Kolom:", df_sesi.columns)
print(df_sesi.head())

Shape: (43, 4)
Kolom: Index(['id_sesi', 'nama_sesi', 'waktu_mulai', 'waktu_selesai'], dtype='object')
  id_sesi        nama_sesi waktu_mulai waktu_selesai
0  S00001    GE/LLC Sesi 1    15:45:00      16:45:00
1  S00002    GE/LLC Sesi 2    17:00:00      18:00:00
2  S00003    GE/LLC Sesi 3    18:15:00      19:15:00
3  S00004   CC Kids Sesi 1    10:10:00      11:10:00
4  S00005  CC Adult Sesi 1    16:00:00      17:00:00


In [18]:
data_loaded['sesi'] = df_sesi

In [19]:
with open("fase_1_afrida.pkl", "wb") as f:
    pickle.dump(data_loaded, f)

In [20]:
data_loaded = pickle.load(open("fase_1_afrida.pkl", "rb"))

In [21]:
data_loaded['topik_diskusi'] = df_topik

In [22]:
with open("fase_1_afrida.pkl", "wb") as f:
    pickle.dump(data_loaded, f)

In [23]:
data_loaded = pickle.load(open("fase_1_afrida.pkl", "rb"))

In [24]:
data_loaded['libur'] = df_libur

In [25]:
with open("fase_1_afrida.pkl", "wb") as f:
    pickle.dump(data_loaded, f)

In [26]:
data_loaded = pickle.load(open("fase_1_afrida.pkl", "rb"))

In [27]:
df_fase1_afrida = {
    "kursus": df_kursus,
    "level": df_level,
    "sesi": df_sesi,
    "libur": df_libur,
    "topik_diskusi": df_topik,
    "kursus_level": df_kursus_level,
    "kursus_libur": df_kursus_libur
}

import pickle
with open("fase_1_afrida.pkl", "wb") as f:
    pickle.dump(df_fase1_afrida, f)
print("✅ pickle disimpan ulang")

✅ pickle disimpan ulang


In [28]:
import pickle

# Pastikan df_fase1_afrida sudah dibuat
# (harusnya sudah dari cell sebelumnya)
if 'df_fase1_afrida' not in dir():
    df_fase1_afrida = {
        "kursus": df_kursus,
        "level": df_level,
        "sesi": df_sesi,
        "libur": df_libur,
        "topik_diskusi": df_topik,
        "kursus_level": df_kursus_level,
        "kursus_libur": df_kursus_libur
    }

# Simpan fase_1_afrida.pkl
with open('fase_1_afrida.pkl', 'wb') as f:
    pickle.dump(df_fase1_afrida, f)
print("✅ fase_1_afrida.pkl disimpan")

# Update df_new.pkl jika ada
try:
    with open('../df_new.pkl', 'rb') as f:
        df_new = pickle.load(f)
    
    df_new['libur'] = df_libur
    df_new['topik_diskusi'] = df_topik
    df_new['kursus_libur'] = df_kursus_libur

    with open('../df_new.pkl', 'wb') as f:
        pickle.dump(df_new, f)
    print("✅ df_new.pkl diperbarui")
except FileNotFoundError:
    print("ℹ️ df_new.pkl tidak ditemukan, skip update")

✅ fase_1_afrida.pkl disimpan
ℹ️ df_new.pkl tidak ditemukan, skip update


In [29]:
from IPython.display import display

for key, df in data_loaded.items():
    print(f"\n📊 {key}")
    display(df.head())


📊 kursus


,id_kursus,nama_kursus,deskripsi
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner"
1,K00002,LEAP - Coding Class,Coding Class Regular
2,K00003,LEAP - Leap Literacy Club,LLC
3,K00004,LEAP - Conversation Class,Conversation Class for Adults
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,"



📊 level


,id_level,nama_level,urutan_level
0,L00001,Balloons 1A,1
1,L00002,Balloons 1B,2
2,L00003,Balloons 1C,3
3,L00004,Balloons 2A,4
4,L00005,Balloons 2B,5



📊 sesi


...

,id_topik_diskusi,topik_diskusi,deskripsi_topik_diskusi
0,None,Kendala Siswa,
1,None,Kendala Kelas,
2,None,Kendala Jadwal,
3,None,Ujian Susulan & Remidi,
4,None,"Kendala Zoom, Class In, Koneksi & Device",



📊 kursus_level


,id_kursus,id_level
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005



📊 kursus_libur


,id_kursus,id_libur
0,K00001,L00070
1,K00001,L00068


## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection